<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/rag_customer_support_vikram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-openai faiss-cpu datasets


In [2]:
from datasets import load_dataset
test_data = load_dataset("galileo-ai/ragbench", "delucionqa", split="test")

print(test_data.column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']


In [4]:
documents = [row["documents"] for row in test_data]
print(len(documents))

184


In [6]:
from langchain.schema import Document as LCDocument

# Flatten the list of lists into a single list of strings
all_doc_strings = [doc_str for doc_list in documents for doc_str in doc_list]

lc_docs = [LCDocument(page_content=doc_str) for doc_str in all_doc_strings]

In [7]:
# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")



/tmp/ipython-input-3134206146.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Vectore Store
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(lc_docs, embedder)



In [9]:
# Create Retriever
retriever = vector_db.as_retriever(search_k=5)


In [10]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

def set_llm(model_repo: str = "google/flan-t5-base"):
    """Set up the language model for generation."""
    load_dotenv()
    openai_model = ChatOpenAI(
        model_name="gpt-3.5-turbo",
        temperature=0.5,
        api_key=os.getenv("OPENAI_API_KEY")
    )
    return openai_model

In [13]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

def set_llm(model_repo: str = "google/flan-t5-base"):
    """Set up the language model for generation."""
    load_dotenv()
    openai_model = ChatOpenAI(
        model_name="gpt-3.5-turbo",
        temperature=0.5,
        api_key=os.getenv("OPENAI_API_KEY")
    )
    return openai_model

def format_docs(docs: list[Document]) -> str:
    """Formats a list of documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

def setup_rag_chain():
  """Set up the RAG chain with prompt template."""
  template = """Answer the question based only on the following context:
  {context}

  Question: {question}
  """
  prompt = ChatPromptTemplate.from_template(template)

  # Define the RAG chain to return a dictionary
  rag_chain = (
      RunnableParallel(
          # Retrieve documents based on the question
          # The 'context' key will hold the list of Document objects
          context=retriever,
          # The 'question' key will hold the original question
          question=RunnablePassthrough()
      )
      | RunnableParallel(
          # Process the retrieved context for the prompt and generate the answer
          result=(
              RunnablePassthrough.assign(context=lambda x: format_docs(x["context"])) # Format docs for prompt
              | prompt
              | set_llm()
              | StrOutputParser()
          ),
          # Pass the original retrieved documents as 'source_documents'
          source_documents=lambda x: x["context"]
      )
  )
  return rag_chain

In [14]:
query = test_data[0]["question"]

rag_chain = setup_rag_chain()
response = rag_chain.invoke(query)

print("Answer:", response["result"])
print("Sources:", response["source_documents"])

Answer: If you fail to securely latch the tailgate, it could result in damage to the vehicle or cargo.
Sources: [Document(id='8eae2a0a-a4c3-41ce-bf70-6b52c78ee056', metadata={}, page_content=' Closing To close the tailgate, lift upward until both sides latch into place.  CAUTION: After closing, pull back on the tailgate firmly to ensure it is securely latched.  Failure to securely latch the tailgate could result in damage to the vehicle or cargo.  Note: If the Tonneau Cover is installed, make sure the Tonneau Cover is fully closed before closing the tailgate.  Due to the presence of the Center High-Mounted Stop Lamp, removal of the tailgate is not recommended.'), Document(id='47449cb4-9101-437d-be20-7adb9cd150f3', metadata={}, page_content=' Closing To close the tailgate, lift upward until both sides latch into place.  CAUTION: After closing, pull back on the tailgate firmly to ensure it is securely latched.  Failure to securely latch the tailgate could result in damage to the vehicle 

In [15]:
print(f" Question, {query}")
print(f" Answer, {response["result"]}")
print(f" Answer, {test_data[0]["response"]}")

 Question, What if I fail to latch the tailgate properly?
 Answer, If you fail to securely latch the tailgate, it could result in damage to the vehicle or cargo.
 Answer, If you fail to securely latch the tailgate properly, it could result in damage to the vehicle or cargo.
